<a href="https://colab.research.google.com/github/PHorley11/Interdisciplinary-GT---Assignment-2/blob/main/assignment_2_gt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("adilshamim8/education-and-career-success")

print("Path to dataset files:", path)

100%|██████████| 3.50k/3.50k [00:00<00:00, 3.79MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/adilshamim8/education-and-career-success/versions/3


In [ ]:
import pandas as pd
import os

files = os.listdir(path)
print(files)
educ = pd.read_csv(os.path.join(path, files[0]))
educ = educ.drop(columns=['Career_Satisfaction', 'Years_to_Promotion','Current_Job_Level','Work_Life_Balance','Entrepreneurship','Student_ID','Age','Gender','High_School_GPA','SAT_Score','Soft_Skills_Score','Networking_Score'])
educ.head()

['education_career_success.csv']


,University_GPA,Field_of_Study,Internships_Completed,Projects_Completed,Certifications,Job_Offers,Starting_Salary
0,3.6,Computer Science,3,7,2,3,85000
1,3.4,Business,2,5,3,2,65000
2,3.8,Engineering,4,9,4,4,120000
3,3.2,Psychology,1,3,1,1,48000
4,3.5,Medicine,2,6,2,3,95000
...,...,...,...,...,...,...,...
395,3.5,Business,3,7,3,3,88000
396,3.3,Law,2,6,2,2,72000
397,3.9,Computer Science,4,9,5,5,140000
398,3.4,Psychology,2,5,2,2,80000


# Calculating α and β

In [ ]:
prop_one_or_more = (educ["Internships_Completed"] >= 1).mean()

print("Proportion with at least 1 internship:", prop_one_or_more)

Proportion with at least 1 internship: 0.975


In this dataset 97.5% of students have completed at least one internship. To meaningfully distinguish candidates and help separate types, we will look at students with two or more internships.

In [ ]:
educ["has_internship"] = (educ["Internships_Completed"] > 2).astype(int) # to indicate whether students have completed more than two internship
educ["hands_on_score"] = educ["Projects_Completed"] + educ["Certifications"] # students will be considered the 'hands-on' type if they focus on completing many projects and on obtaining certifications

gpa_threshold = educ["University_GPA"].median()
hands_on_threshold = educ["hands_on_score"].median()

print(gpa_threshold, hands_on_threshold)

3.45 9.0


Students need to have a 3.45 or higher GPA to be considered as the 'academic' type and have to have more than 9 projects and certifications to be considered the 'hands-on' type.

In [ ]:
educ["gpa_norm"] = (educ["University_GPA"] - educ["University_GPA"].mean()) / educ["University_GPA"].std()
educ["hands_on_norm"] = (educ["hands_on_score"] - educ["hands_on_score"].mean()) / educ["hands_on_score"].std()

def classify(row):
    if row["hands_on_norm"] > row["gpa_norm"]:
        return "hands_on"
    else:
        return "academic"

educ["type"] = educ.apply(classify, axis=1)
educ.head()

,University_GPA,Field_of_Study,Internships_Completed,Projects_Completed,Certifications,Job_Offers,Starting_Salary,has_internship,hands_on_score,type,gpa_norm,hands_on_norm
0,3.6,Computer Science,3,7,2,3,85000,1,9,academic,0.556875,0.044733
1,3.4,Business,2,5,3,2,65000,0,8,academic,-0.143597,-0.274789
2,3.8,Engineering,4,9,4,4,120000,1,13,hands_on,1.257346,1.322820
3,3.2,Psychology,1,3,1,1,48000,0,4,academic,-0.844068,-1.552875
4,3.5,Medicine,2,6,2,3,95000,0,8,academic,0.206639,-0.274789


In [ ]:
educ_clean = educ[educ["type"] != "mixed"]

educ_clean["type"].value_counts()

,count
type,
hands_on,206
academic,194


In [ ]:
alpha = educ[educ["type"] == "hands_on"]["has_internship"].mean()
beta = educ[educ["type"] == "academic"]["has_internship"].mean()

print("alpha (hands-on -> internship):", alpha)
print("beta (academic -> internship):", beta)

alpha (hands-on -> internship): 0.6990291262135923
beta (academic -> internship): 0.3556701030927835


# Supporting data

In [ ]:
educ.groupby("has_internship")["Job_Offers"].mean()

,Job_Offers
has_internship,
0,1.529412
1,3.802817


Students with 2 or more internships get on average 3.8 job offers while students with less than 2 internships get on average 1.5 offers.

In [ ]:
educ.groupby("has_internship")["Starting_Salary"].mean()

,Starting_Salary
has_internship,
0,63294.117647
1,108868.544601


Students who have two or more internships have a higher starting salary ($108,868) compared to those with less than two internships ($$63,294)

In [ ]:
educ.groupby(["Field_of_Study", "has_internship"])["Starting_Salary"].mean()

Field_of_Study    has_internship
Arts              0                  46093.750000
Business          0                  63875.000000
                  1                 103500.000000
Computer Science  0                  75000.000000
                  1                 130106.382979
Education         0                  48000.000000
Engineering       0                  75090.909091
                  1                 104722.222222
Finance           0                  75000.000000
                  1                  95000.000000
Law               0                  74769.230769
                  1                  89833.333333
Marketing         0                  67111.111111
                  1                  89461.538462
Medicine          0                  78500.000000
                  1                 126060.606061
Nursing           0                  67000.000000
Psychology        0                  59459.459459
                  1                  89111.111111
Name: Starting_Salary, dtype: float64

Within the same field of study, students on average have higher starting salaries if they have completed two or more internships.

In [ ]:
academic = educ[educ["type"] == "academic"]

academic.groupby("has_internship")[["Job_Offers", "Starting_Salary"]].mean()

,Job_Offers,Starting_Salary
has_internship,,
0,1.440000,62528.000000
1,3.797101,107463.768116


Within the academic type, those who signal experience through two or more internships have on average more job offers (1.44 vs 3.79) and a higher starting salary (62,528 vs 107,463).

In [ ]:
hands_on = educ[educ["type"] == "hands_on"]

hands_on.groupby("has_internship")[["Job_Offers", "Starting_Salary"]].mean()

,Job_Offers,Starting_Salary
has_internship,,
0,1.709677,64838.709677
1,3.805556,109541.666667


Within the hands-on type, those who signal experience through two or more internships have on average more job offers (1.7 vs 3.8) and a higher starting salary (64,838 vs 109,541).

In [ ]:
educ.groupby(["type", "has_internship"])[["Job_Offers", "Starting_Salary"]].mean()

Job_Offers  Starting_Salary
type     has_internship                             
academic 0                 1.440000     62528.000000
         1                 3.797101    107463.768116
hands_on 0                 1.709677     64838.709677
         1                 3.805556    109541.666667

Between both types, they are each penalised by not signaling experience and benefit from signaling experience. Since it has a similar effect on both, this supports the importance of signaling in the job market over type.